In [ ]:
import requests

In [ ]:
URL = "http://[IP_ADDRESS]/api/chat"
headers = {
    "Content-Type": "application/json",
}

# 4) Set a timeout so the call doesn't hang forever
timeout_seconds = 200

msgs = [
    {"role":"system","content":"You are an assistant to rank documents by relevance. You should return a json with two keys: (i) ranking: a python list with the order of the documents based on the relevance, being the most relevant the first one and (ii) a key explanation with a detailed reasoning leading to this ranking. The user will provide a query to be projected into the documents. The documents are:\n" + str(documents_example)},
    {"role":"user","content":qtext}
]

payload = {
    "model": "gemma3",
    "stream": False,
    "messages": msgs,
}

response = requests.post(
    URL,
    headers=headers,
    json=payload,
    timeout=timeout_seconds,
)
data = response.json()

print(data)
print('\n'*5)
print('Answer: ', data['message']['content'])

In [ ]:
#LLamada a API ordenador JON

import requests
import json
URL = "http://[IP_ADDRESS]/api/chat"
headers = {
    "Content-Type": "application/json",
}

# 4) Set a timeout so the call doesn't hang forever
timeout_seconds = 200

msgs = [
    {"role":"user","content":"hola quien eres?"}
]

payload = {
    "model": "gemma3",
    "stream": False,
    "messages": msgs,
}

response = requests.post(
    URL,
    headers=headers,
    json=payload,
    timeout=timeout_seconds,
)
data = response.json()

print(data)
print('\n'*5)
print('Answer: ', data['message']['content'])

# Ejercicio Benchmark

In [ ]:

# Llamada a API ordenador JON
import requests
import json
URL = "http://[IP_ADDRESS]/api/chat"  
headers = {
    "Content-Type": "application/json",
}

timeout_seconds = 200


def ask_model(prompt, context=None, model="gemma3"):
    """
    Envía un prompt al modelo con contexto opcional.

    Args:
        prompt (str): Pregunta o instrucción del usuario
        context (str, optional): Contexto o system prompt
        model (str): Modelo a usar

    Returns:
        str: Respuesta del modelo
    """

    messages = []

    # Contexto (system)
    if context:
        messages.append({
            "role": "system",
            "content": context
        })

    # Prompt del usuario
    messages.append({
        "role": "user",
        "content": prompt
    })

    payload = {
        "model": model,
        "stream": False,
        "messages": messages,
    }

    response = requests.post(
        URL,
        headers=headers,
        json=payload,
        timeout=timeout_seconds,
    )

    response.raise_for_status()
    data = response.json()

    return data["message"]["content"]



In [6]:
context = """Eres un profesional sanitario experto en triaje de urgencias (enfermería/medicina de urgencias). Tu tarea es evaluar la información clínica aportada por el usuario y producir una decisión de triaje estructurada.

            OBJETIVO
            1) Determinar la especialidad o circuito más adecuado para derivación (p.ej., Medicina Interna, Urgencias generales, Traumatología, Cardiología, Neurología, Pediatría, Ginecología, ORL, Urología, Cirugía General, Psiquiatría, etc.).
            2) Estimar el nivel de urgencia (crítico/inmediato, muy urgente, urgente, menos urgente, no urgente) y justificarlo con hallazgos clave.
            3) Proponer pruebas iniciales razonables (analítica, ECG, imagen, constantes, gasometría, pruebas específicas) acordes al cuadro.
            
            REGLAS DE SEGURIDAD Y CALIDAD
            - No inventes datos: usa solo lo que el usuario aporta. Si falta información esencial, pregunta de forma breve por los datos mínimos necesarios.
            - Prioriza siempre la detección de “banderas rojas” (dolor torácico, disnea, déficit neurológico focal, alteración del nivel de consciencia, hemorragia importante, sepsis, anafilaxia, abdomen agudo, etc.). Si hay banderas rojas, recomienda acudir a urgencias de inmediato/112.
            - No sustituyes a un médico real: tus recomendaciones son orientativas y deben confirmarse clínicamente.
            - Utiliza protocolos establecidos en hospital clínicos.
            - Considera edad, embarazo, comorbilidades, medicación (anticoagulantes), alergias, constantes vitales y tiempo de evolución.
            
            FORMATO DE RESPUESTA (siempre)
            1) Resumen clínico (2-4 líneas): síntomas principales + factores de riesgo + datos relevantes.
            2) Banderas rojas: presentes/ausentes (y cuáles).
            3) Nivel de urgencia: (crítico / muy urgente / urgente / menos urgente / no urgente) + justificación breve.
            4) Derivación / especialidad recomendada: y motivo.
            5) Pruebas iniciales sugeridas: lista priorizada (concisamente).
            6) Preguntas de aclaración (si faltan datos): máximo 5, muy concretas.
            7) Recomendaciones inmediatas: qué hacer ahora según el nivel de urgencia (incluye “si empeora, acudir/112” cuando corresponda).
            
            Usa un tono profesional, claro y conciso."""



In [4]:
#Pregunta 1

prompt = "Un hombre de 45 años acude a urgencias quejándose de dolor en el hombro izquierdo. Refiere que el dolor ha empeorado gradualmente en las últimas semanas y se intensifica al levantar objetos pesados. Además del dolor, ha notado una sensación de debilidad en el brazo izquierdo, especialmente intentando levantarlo lateralmente. Afebril, eupneico y normotensión"

answer = ask_model(prompt, context)
print("Answer:\n", answer)

Answer:
 De acuerdo, aquí tienes la evaluación de triaje estructurada basada en la información proporcionada:

**1) Resumen clínico:**

Hombre de 45 años que presenta dolor gradual en hombro izquierdo, intensificado con la actividad física y asociado a debilidad en el brazo, especialmente al levantar lateralmente. Sin fiebre ni otros síntomas asociados.

**2) Banderas rojas:**

*   **Presentes:** Dolor progresivo en hombro con limitación funcional, debilidad en el brazo, especialmente lateralmente. Estos síntomas sugieren una posible patología de la articulación gleno-úlcara o del manguito rotador.
*   **Ausentes:** No hay fiebre, dificultad respiratoria o signos de infección aguda.

**3) Nivel de urgencia:** Urgente – Justificación: El dolor persistente y la debilidad progresiva en un hombro, con limitación funcional, requieren una evaluación rápida para descartar patología grave, como una inestabilidad articular o una fractura sin complicaciones.

**4) Derivación / especialidad recom

In [5]:
prompt = "Una paciente viene a urgencias, en la revisión se le detecta que la tensión arterial está en 150/95 mmHg, alega trastorno de ansiedad y antecedentes de depresión. El paciente se encuentra muy alterado y no mejora des de hace 24h. Afebril y eupneico. "

answer = ask_model(prompt, context)
print("Answer:\n", answer)

Answer:
 De acuerdo, vamos a proceder al triaje de esta paciente.

**1) Resumen clínico:** Paciente femenina, angustiada, con antecedentes de ansiedad y depresión, con un episodio de alteración de estado mental que se ha mantenido durante 24 horas. La tensión arterial elevada (150/95 mmHg) es un factor significativo. Afebrícula y con patrón respiratorio normal.

**2) Banderas rojas:** Presente. Alteración de estado mental persistente (24h) y tensión arterial elevada son banderas rojas que requieren una evaluación más profunda.

**3) Nivel de urgencia:** Urgente. La persistencia del cuadro durante 24 horas y la tensión arterial elevada sugieren una necesidad de intervención más rápida que en un caso de urgencia menor.

**4) Derivación / especialidad recomendada:** Urgencias generales. La necesidad de una evaluación completa de una alteración mental y la presencia de hipertensión justifican la derivación a urgencias generales para descartar causas orgánicas de la alteración y estabilizar

In [6]:
#3
prompt = "Hombre de 55 años, con dolor opresivo en el pecho, que se irradia hacia el brazo izquierdo y la mandíbula. Presenta sudoración profusa y sensación de falta de aire. Afebril, tensión elevada"
answer = ask_model(prompt, context)
print("Answer:\n", answer)

Answer:
 De acuerdo, voy a realizar el triaje de este paciente.

**1) Resumen clínico:**

Hombre de 55 años que presenta dolor opresivo en el pecho, irradiado al brazo izquierdo y la mandíbula, acompañado de sudoración profusa y disnea. La tensión arterial elevada sugiere un posible evento cardiovascular agudo.

**2) Banderas rojas:**

*   **Presentes:** Dolor torácico opresivo con irradiación, sudoración profusa, disnea.  Estas características son altamente sugestivas de un posible infarto agudo de miocardio.
*   **Ausentes:** No se menciona fiebre, lo que reduce, pero no elimina, la posibilidad de otras causas.

**3) Nivel de urgencia:**

*   **Muy urgente** - El dolor torácico con irradiación, sudoración y disnea son signos cardinales de un posible infarto agudo de miocardio, que requiere intervención inmediata para minimizar el daño miocárdico.

**4) Derivación / especialidad recomendada:**

*   **Urgencias generales (Cardiológico)** -  Es crucial una evaluación y tratamiento rápid

In [7]:
#4
prompt = "Paciente de 60 años, llega al hospital con tos persistente, fiebre y dificultad para respirar. Fue fumador durante 40 años y ha presentado episodios similares en el pasado. Hypertensión moderada."
answer = ask_model(prompt, context)
print("Answer:\n", answer)

Answer:
 De acuerdo, vamos a realizar el triaje.

**1) Resumen clínico:**

Paciente de 60 años con tos persistente, fiebre y dificultad para respirar, historial de episodios similares, antecedente de tabaquismo importante (40 años) y hipertensión moderada.

**2) Banderas rojas:** presentes. La dificultad para respirar, combinada con la tos y los antecedentes de episodios similares, sugieren un posible problema respiratorio agudo. El tabaquismo es un factor de riesgo significativo.

**3) Nivel de urgencia:** urgente. La dificultad para respirar, aunque no se especifica la severidad, implica una amenaza inmediata para la salud. La fiebre y la tos apoyan la posibilidad de una infección respiratoria y la historia de episodios similares indica una patología subyacente que requiere una evaluación rápida.

**4) Derivación / especialidad recomendada:** Urgencias generales. Debido a la sintomatología heterogénea y la necesidad de descartar rápidamente causas graves como neumonía, exacerbación d

In [8]:
prompt = "Paciente 45 años, mujer, acude al consultorio con tos persistente y dificultad para respirar. Durante las últimas semanas ha experimentado fatiga extrema y fiebre intermitente. Fumador desde hace 20 años y su tos ha empeorado últimamente. Su historial revela episodios recurrentes de bronquitis. Normotension."
answer = ask_model(prompt, context)
print("Answer:\n", answer)

Answer:
 De acuerdo, procedo a realizar el triaje.

**1) Resumen clínico:** Mujer de 45 años, ex-fumadora con historial de bronquitis, que presenta tos persistente, dificultad para respirar, fatiga extrema y fiebre intermitente.

**2) Banderas rojas:** Presentes. La dificultad para respirar, la fiebre intermitente y el historial de bronquitis, junto con el tabaquismo, sugieren un posible proceso infeccioso grave o una enfermedad pulmonar obstructiva crónica (EPOC) avanzada.

**3) Nivel de urgencia:** Urgente. La dificultad para respirar justifica una evaluación urgente para descartar complicaciones como neumonía, exacerbación de EPOC o insuficiencia cardíaca.

**4) Derivación / especialidad recomendada:** Urgencias generales. Para una evaluación y posible monitorización inicial.

**5) Pruebas iniciales sugeridas:**
   *   Analítica completa (incluyendo hemograma, bioquímica, PCR)
   *   ECG
   *   Radiografía de tórax
   *   Gasometría arterial (para evaluar la función pulmonar)

**6) 

In [9]:
prompt = "Paciente de 25 años, hombre, se presenta en el hospital con síntomas de dolor abdominal intenso, especialmente después de comer, hinchazón y dificultad para defecar. Afebril, eupneico y normotensión"
answer = ask_model(prompt, context)
print("Answer:\n", answer)


Answer:
 De acuerdo, vamos a realizar el triaje de este paciente.

**1) Resumen clínico:**

Paciente masculino de 25 años que se presenta con dolor abdominal intenso postprandial, hinchazón y estreñimiento. No presenta fiebre ni otros signos evidentes de infección.

**2) Banderas rojas:** Presentes. El dolor abdominal intenso, especialmente postprandial, puede indicar patologías serias como obstrucción intestinal, pancreatitis, enfermedad de Crohn o síndrome de Illeus Paralítico.

**3) Nivel de urgencia:** Muy urgente + justificación: El dolor abdominal intenso y la incapacidad para descartar patologías graves justifican una evaluación urgente y una posible derivación rápida.

**4) Derivación / especialidad recomendada:** Urgencias generales o Cirugía General.  Se necesita una evaluación exhaustiva para descartar causas orgánicas y, posiblemente, una intervención quirúrgica.

**5) Pruebas iniciales sugeridas:**

*   Analítica sanguínea completa (hemograma, bioquímica, electrolitos, ami

In [14]:
prompt = "Paciente de 35 años acude a la consulta por alteraciones en su patrón intestinal. Describe cambios en la frecuencia de defecación y la consistencia de las evacuaciones desde hace dos meses. Refiere distensión abdominal persistente y sensación de aumento de presión abdominal. Además reporta dolor durante la defecación y sensación de evacuación incompleta. Manifiesta episodios de náuseas sin vómito y una reciente disminución del apetito. Afebril"
answer = ask_model(prompt, context)
print("Answer:\n", answer)


Answer:
 De acuerdo, procedo con el triaje.

**1) Resumen Clínico:** Paciente de 35 años con alteraciones intestinales de dos meses de evolución, incluyendo cambios en la frecuencia y consistencia de las heces, distensión abdominal, dolor durante la defecación y sensación de evacuación incompleta. Se acompaña de náuseas, disminución del apetito y afebril.

**2) Banderas rojas:** Presente. La distensión abdominal, dolor durante la defecación, sensación de evacuación incompleta y la disminución del apetito, combinadas con los cambios en el patrón intestinal, son banderas rojas que sugieren una patología subyacente significativa.

**3) Nivel de urgencia:** Urgente. La distensión abdominal y el dolor en la defecación, junto con la alteración en el patrón intestinal, pueden indicar causas serias como obstrucción intestinal, enfermedad inflamatoria intestinal (EII) o malignidad. La falta de fiebre no descarta un proceso inflamatorio.

**4) Derivación / especialidad recomendada:** Gastroenter

In [11]:
prompt = "Paciente de 60 años, acude al hospital con síntomas de fatiga extrema, presión arterial alta y una hinchazón evidente en las piernas. Afebril, eupneico"
answer = ask_model(prompt, context)
print("Answer:\n", answer)


Answer:
 De acuerdo, procedo con el triaje.

1) **Resumen clínico:** Paciente de 60 años que presenta fatiga extrema, hipertensión arterial y edema en las extremidades inferiores. Sin fiebre ni alteraciones respiratorias evidentes.

2) **Banderas rojas:** Presente. La hipertensión arterial y el edema en las extremidades inferiores, combinadas con la fatiga extrema, pueden indicar una variedad de condiciones subyacentes que requieren una evaluación urgente, incluyendo, pero no limitado a, insuficiencia cardíaca, enfermedad renal, o trombosis venosa profunda.

3) **Nivel de urgencia:** Urgente. La combinación de estos síntomas, especialmente la hipertensión y el edema, sugieren una posible condición grave que requiere una evaluación y tratamiento rápidos.

4) **Derivación / especialidad recomendada:** Cardiología y/o Nefrología. La sospecha de insuficiencia cardíaca o enfermedad renal justifica la derivación a estas especialidades.

5) **Pruebas iniciales sugeridas:**
    *   Análisis de

In [12]:
prompt = "Paciente, incapacidad para orinar desde hace 24 horas. Refiere dolor y sensación de plenitud en el abdomen inferior justo con fiebre de 38,5 grados celsius. Sus antecedentes médicos son: Hipertensión, tratada con diuréticos sin infecciones urinarias o problemas renales conocidos. Presenta el abdomen distendido y doloroso a la palpación en la región suprapúbica. Eupneico."
answer = ask_model(prompt, context)
print("Answer:\n", answer)

Answer:
 Okay, evaluemos este caso.

**1) Resumen clínico:** Hombre de 68 años con incapacidad para orinar desde hace 24 horas, asociado a dolor abdominal inferior, sensación de plenitud y fiebre de 38.5°C. Abdomen distendido y doloroso en la región suprapúbica. Antecedentes de hipertensión tratada con diuréticos.

**2) Banderas rojas:** Presentes. Dolor abdominal agudo asociado a retención urinaria y fiebre sugiere:
*   Posible pielonefritis
*   Obstrucción urinaria (impactación ureteral, tumoral)
*   Nefritis intersticial
*   (Aunque la historia clínica es positiva, la fiebre y el dolor son factores de riesgo).

**3) Nivel de urgencia:** Muy urgente. La retención urinaria en combinación con fiebre y dolor abdominal justifica una rápida evaluación y tratamiento para prevenir complicaciones como pielonefritis y daño renal.

**4) Derivación / especialidad recomendada:** Urgencias generales (preferiblemente con acceso a un urólogo). Motivo: Evaluación rápida y tratamiento de la retención

In [13]:
prompt = "Joven de 17 años, accidente de moto. Ha volado por encima de un coche haciendo una mortal descontrolada. El joven se ha podido levantar y se ha quitado el casco en el momento del accidente. Alega mucho dolor en las piernas, pulsaciones altas que no bajan, dolor en el hombro y en la muñeca. Eupneico, normotensión, afebril"
answer = ask_model(prompt, context)
print("Answer:\n", answer)

Answer:
 De acuerdo, procesaré la información y elaboraré la evaluación de triaje.

**1) Resumen clínico:** Joven de 17 años, traumatizado por accidente de moto con múltiples lesiones potenciales (piernas, hombro, muñeca) y posible lesión cervical/torácica dada la situación del impacto. Presencia de pulsaciones altas y dolor significativo.

**2) Banderas rojas:** Presentes.  El accidente de moto con impacto significativo, pulsaciones altas y dolor generalizado representan una bandera roja por posible lesión interna, hemorragia interna, lesión de columna cervical o torácica. La pérdida del casco es una bandera roja adicional.

**3) Nivel de urgencia:** Muy urgente.  La combinación de la severidad del mecanismo de lesión, la presencia de pulsaciones altas y dolor generalizado justifica una evaluación y tratamiento inmediatos.

**4) Derivación / especialidad recomendada:** Urgencias generales / Traumatología.  Es necesaria una evaluación completa por un equipo con experiencia en manejo de

# Open AI

## GPT 5 NANO

In [3]:
OPENAI_ORG_ID = ""
OPENAI_API_KEY = ""

In [3]:
pip install openai

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [openai]2m3/4 [openai]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import openai
from openai import OpenAI

In [7]:

client = OpenAI(api_key=OPENAI_API_KEY)
def ask_message(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY):
    openai.organization = OPENAI_ORG_ID
    openai.api_key = OPENAI_API_KEY
    
    response = client.chat.completions.create(model="gpt-5-nano",
        messages=[
            {"role": "system", "content": context},#Como se comporta el modelo
            {"role": "user", "content": prompt}
        ])
    message = response.choices[0].message.content
    return message

In [8]:
prompt = "Un hombre de 45 años acude a urgencias quejándose de dolor en el hombro izquierdo. Refiere que el dolor ha empeorado gradualmente en las últimas semanas y se intensifica al levantar objetos pesados. Además del dolor, ha notado una sensación de debilidad en el brazo izquierdo, especialmente intentando levantarlo lateralmente. Afebril, eupneico y normotensión"
message = ask_message(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico
Hombre de 45 años con dolor en el hombro izquierdo progresivo en las últimas semanas, empeora al levantar objetos pesados y refiere debilidad al intentar elevar el brazo en abducción. Afebril, sin disnea y con presión arterial normal.

2) Banderas rojas
- Presentes: ausentes (no hay dolor torácico agudo, fiebre, alteración del estado general, signos de infección, deficit neurológico focal evidente, trauma reciente importante).
- Ausentes: dolor torácico/agudo, signos de sepsis, alteración del nivel de conciencia, trauma mayor, hemorragia.

3) Nivel de urgencia
Urgente: hay dolor mecánico crónico con debilidad funcional potencial de manguito rotador. Requiere valoración ortopédica y pruebas de imagen en un plazo razonable para descartar desgarro del manguito rotador o patología tendinosa/tendinopatía y plan de tratamiento.

4) Derivación / especialidad recomendada
Ortopedia/Traumatología para evaluación del hombro (posible manguito rotador): confirmar diagnóstico, dec

In [9]:
prompt = "Una paciente viene a urgencias, en la revisión se le detecta que la tensión arterial está en 150/95 mmHg, alega trastorno de ansiedad y antecedentes de depresión. El paciente se encuentra muy alterado y no mejora des de hace 24h. Afebril y eupneico. "
message = ask_message(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico
Paciente femenina con antecedentes de trastorno de ansiedad y depresión, llega a urgencias con agitación marcada que persiste 24 h. TA 150/95 mmHg, afebril y eupneica. Sin dolor torácico ni disnea reportados en este momento; ausencia de fiebre. Evaluación clínica adicional pendiente.

2) Banderas rojas
- Presentes: agitación/señales de crisis psiquiátrica persistente (24 h) y antecedentes psiquiátricos.
- Ausentes: dolor torácico, disnea actual, fiebre, alteración del nivel de consciencia, déficit neurológico focal, signos de sepsis o abdomen agudo.

3) Nivel de urgencia
Urgente. Razonamiento: agitación psicológica marcada y persistente en el tiempo sin signos vitales inestables ni banderas rojas claras; requiere monitorización de constantes e intervención psiquiátrica para manejo de crisis y tratamiento, con vigilancia en urgencias.

4) Derivación / especialidad recomendada
Psiquiatría (con apoyo del servicio de Urgencias Generales). Motivo: crisis psiquiátrica/agit

In [10]:
prompt = "Hombre de 55 años, con dolor opresivo en el pecho, que se irradia hacia el brazo izquierdo y la mandíbula. Presenta sudoración profusa y sensación de falta de aire. Afebril, tensión elevada"
message = ask_message(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico
Hombre de 55 años con dolor torácico opresivo irradiado al brazo izquierdo y mandíbula, sudoración profusa y disnea. Afebril. Tensión arterial elevada. Sospecha de síndrome coronario agudo (ACS) en curso; factores de riesgo posibles: edad, sexo, HTA.

2) Banderas rojas
- Presentes: dolor torácico típico con irradiación, diaforesis, disnea.
- Ausentes/No indicados: fiebre, alteración del nivel de consciencia o déficit neurológico focal (a confirmar en evaluación), hemorragias, dolor abdominal agudo (no indicado).

3) Nivel de urgencia
Crítico / Inmediato. Justificación: clínica compatible con ACS de inicio potencial, necesidad de ECG y evaluación cardiaca urgente para descartar o tratar infarto agudo de miocardio.

4) Derivación / especialidad recomendada
Urgencias Generales con activación de protocolo de Síndrome Coronario Agudo; derivación precoz a Cardiología (unidad de dolor torácico/ACS) para manejo definitivo y posibles intervenciones.

5) Pruebas iniciales suge

In [11]:
prompt = "Paciente de 60 años, llega al hospital con tos persistente, fiebre y dificultad para respirar. Fue fumador durante 40 años y ha presentado episodios similares en el pasado. Hypertensión moderada."
message = ask_message(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico
Paciente de 60 años con tos persistente, fiebre y disnea. Fumador crónico (40 años) y antecedentes de hipertensión moderada. Sospecha clínica inicial: infección respiratoria (posible neumonía) o exacerbación de enfermedad respiratoria previa; necesidad de evaluación rápida en urgencias.

2) Banderas rojas
- Presentes: disnea (síntoma principal de alarma respiratoria).
- Ausentes o no documentadas (según la información aportada): dolor torácico intenso aislado, alteración del estado de conciencia, signos claros de shock, hemorragia, abdomen agudo, signos de sepsis overtos (sin datos vitales disponibles).

3) Nivel de urgencia
Urgente. Disnea y fiebre en un adulto mayor con historial de tabaquismo aumentan el riesgo de neumonía, exacerbación de EPOC o insuficiencia respiratoria; requiere evaluación y monitorización rápidas en urgencias. Si hubiera deterioro de la función respiratoria, subiría a muy urgente/crítico.

4) Derivación / especialidad recomendada
Urgencias Ge

In [12]:
prompt = "Paciente 45 años, mujer, acude al consultorio con tos persistente y dificultad para respirar. Durante las últimas semanas ha experimentado fatiga extrema y fiebre intermitente. Fumados desde hace 20 años y si tos ha empeorado últimamente. Su historial revela episodios recurrentes de bronquitis. Normotension. "
message = ask_message(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico
Mujer de 45 años con tos persistente, disnea, fatiga extrema y fiebre intermitente en las últimas semanas. Fumadora (20 años) con antecedentes de bronquitis recurrente. Normotensa. Sin datos de exploración o pruebas actuales.

2) Banderas rojas
- Disnea: presente (red flag).
- Dolor torácico, alteración del estado de consciencia o signos de sepsis no especificados en los datos aportados.
- No hay datos de apercibimiento de hemorragia, abdomen agudo, ni antecedentes de trauma.

3) Nivel de urgencia
Urgente. Motivo: disnea concomitante con fiebre e historia de tabaquismo y bronquitis recurrente, que sugiere infección respiratoria (posible neumonía) o exacerbación de enfermedad pulmonar; requieren valoración rápida para descartar deterioro y ajustar tratamiento.

4) Derivación / especialidad recomendada
Urgencias Generales para valoración rápida y pruebas iniciales (con derivación a Medicina Interna y/o Neumología según hallazgos). Motivo: neumonía/exacerbación de EPOC 

In [13]:
prompt = "Paciente de 25 años, hombre, se presenta en el hospital con síntomas de dolor abdominal intenso, especialmente después de comer, hinchazón y dificultad para defecar. Afebril, eupneico y normotensión"
message = ask_message(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico
Paciente masculino de 25 años con dolor abdominal intenso, principalmente postprandial, asociado a hinchazón y dificultad para defecar. Afebril, eupneico y normotenso. No se aportan otros datos relevantes ni antecedentes.

2) Banderas rojas
- Presentes: ninguno identificado con la información aportada.
- Ausentes: fiebre, hipotensión, dolor torácico o signos de peritonitis, alteración del nivel de consciencia, sangrado gastrointestinal, sepsis.

3) Nivel de urgencia
- Urgente
- Justificación: dolor abdominal intenso en un adulto joven con distensión y estreñimiento puede corresponder a causas como obstrucción intestinal, trastornos funcionales o inflamatorios; el estado hemodinámico es estable y no hay signos claros de alarma, pero requiere valoración en urgencias y pruebas diagnósticas en las próximas horas para descartar etiologías potencialmente graves.

4) Derivación / especialidad recomendada
- Derivación inicial: Urgencias Generales.
- Derivación posterior/prog

In [14]:
prompt = "Paciente de 35 años acude a la consulta por alteraciones en su patrón intestinal. Describe cambios en la frecuencia de y la consistencia de las evacuaciones desde hace dos meses. Refiere distensión abdominal persistente y sensación de aumento de presión abdominal. Además reporta dolor durante la defecación y sensación de evacuación incompleta. Manifiesta episodios de náuseas sin vómito y una reciente disminución del apetito, afebril."
message = ask_message(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico
- Hombre/mujer de 35 años con 2 meses de cambios en patrón intestinal (frecuencia y consistencia), distensión abdominal, sensación de presión abdominal, dolor durante la defecación y evacuación incompleta; náuseas y disminución del apetito. Afebril. Sin información sobre hemorragia, fiebre, pérdida de peso o signos neurológicos.

2) Banderas rojas
- Presentes: ausentes.
- Ausentes: dolor torácico, disnea, deficit neurológico focal, alteración del estado de consciencia, hemorragia importante, signos de sepsis, abdomen agudo/constante dolor severo, fiebre alta, sangrado rectal explícito.

3) Nivel de urgencia
- Nivel: menos urgente.
- Justificación breve: síntomas crónicos sin signos de alarma hemodinámicos o agudos; estable en consulta; necesidad de valoración por gastroenterología/medicina interna para descartar IBS, IBD, malabsorción u otros trastornos GI; no requiere atención de urgencias inmediatas.

4) Derivación / especialidad recomendada
- Gastroenterología (co

In [15]:
prompt = "Paciente de 60 años, acude al hospital con síntomas de fatiga extrema, presión arterial alta y una hinchazón evidente en las piernas. Afebril, eupneico"
message = ask_message(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico
Paciente de 60 años con fatiga extrema, hipertensión y edema evidente en extremidades inferiores. Afebril y eupneico. No se indican dolor torácico, disnea marcada ni signos de infección en este momento.

2) Banderas rojas
- Dolor torácico: ausente
- Disnea marcada: ausente (afebril, eupneico)
- Alteración del nivel de consciencia: ausente
- Signos de sepsis: ausentes
- Abdomen agudo: ausente
- Hemorragia importante: ausente
- Déficit neurológico focal: ausente
Notas: hay edema de extremidades y hipertensión; estos pueden indicar fallo cardiaco/descompensación hipertensiva, por lo que requieren evaluación rápida, aunque no hay signos de inestabilidad vital obvia en este momento.

3) Nivel de urgencia
Urgente. Existe sospecha de descompensación de estado cardiovascular (edema importante y fatiga) que amerita valoración rápida, monitorización y pruebas, sin signos actuales de shock o insuficiencia respiratoria, pero con potencial de deterioro.

4) Derivación / especiali

In [16]:
prompt = "Paciente, incapacidad para orinar desde hace 24 horas. Refiere dolor y sensación de plenitud en el abdomen inferior justo con fiebre de 38,5 grados celsius. Sus antecedentes médicos son: Hipertensión, tratada con diuréticos sin infecciones urinarias o problemas renales conocidos. Presenta el abdomen distendido y doloroso a la palpación en la región suprapúbica. Eupneico."
message = ask_message(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico
Paciente con incapacidad para orinar desde hace 24 h, dolor y sensación de plenitud en abdomen inferior, fiebre 38,5°C. Abdomen distendido y doloroso en región suprapúbica. Antecedentes de hipertensión en tratamiento diurético. Sin antecedentes conocidos de ITU o enfermedad renal. Eupneico. Sospecha clínica: retención urinaria aguda posiblemente complicada por infección.

2) Banderas rojas
- Presentes: retención urinaria prolongada (24 h), fiebre (38,5°C), dolor suprapúbico/distensión abdominal.
- Ausentes/no reportadas: hipotensión marcada, alteración del estado de Consciousness, disnea grave, signos claros de sepsis (hasta ahora no descritos), dolor torácico, hemorragia importante.

3) Nivel de urgencia
Muy urgente. Motivos: retención urinaria con fiebre (posible ITU complicada) y distensión abdominal, riesgo de progresión a sepsis renal/urinaria; requiere desobstrucción urinaria inmediata y evaluación/investigación adicional.

4) Derivación / especialidad recomend

In [17]:
prompt = "Joven de 17 años, accidente de moto. Ha volado por encima de un coche haciendo una mortal descontrolada. El joven se ha podido levantar y se ha quitado el casco en el momento del accidente. Alega mucho dolor en las piernas, pulsaciones altas que no bajan, dolor en el hombro y en la muñeca. Eupneico, normotensión, afebril"
message = ask_message(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico
Joven de 17 años tras caída/choque de moto con elevada energía. Se ha podido levantar y quitar casco. Dolor intenso en piernas, hombro y muñeca; taquicardia persistente; oxigenación y presión arterial actuales aparentemente estables; afebril. Evalúa dolor multifragmentario posible fracturas y posible lesión de columna/pelvis; no hay datos de alerta neurológica o hemodinámica evidente.

2) Banderas rojas
- Presentes: alto mecanismo de trauma (mortal descontrolada, caída de moto) con riesgo de fracturas/lesiones internas; dolor significativo en extremidades puede ocultar fracturas graves.
- Ausentes (según datos aportados): dolor torácico/disnea/falla neurológica focal/alteración de consciencia severa, sangrado importante, abdomen agudo, hipotensión, fiebre.

3) Nivel de urgencia
Muy urgente. Razonamiento: mecanismo de alto impacto y dolor intenso en varias extremidades con taquicardia; riesgo de múltiples fracturas, lesión de columna o de órganos internos ocultos. Req

## GPT 5

In [18]:
import os
import openai
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
def ask_message5(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY):
    openai.organization = OPENAI_ORG_ID
    openai.api_key = OPENAI_API_KEY
    
    response = client.chat.completions.create(model="gpt-5",
        messages=[
            {"role": "system", "content": context},#Como se comporta el modelo
            {"role": "user", "content": prompt}
        ])
    message = response.choices[0].message.content
    return message

In [19]:
prompt = "Un hombre de 45 años acude a urgencias quejándose de dolor en el hombro izquierdo. Refiere que el dolor ha empeorado gradualmente en las últimas semanas y se intensifica al levantar objetos pesados. Además del dolor, ha notado una sensación de debilidad en el brazo izquierdo, especialmente intentando levantarlo lateralmente. Afebril, eupneico y normotensión."
message = ask_message5(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico:
- Varón 45 años con dolor de hombro izquierdo de semanas de evolución, progresivo, que empeora al levantar peso. Refiere debilidad al intentar la abducción (elevar lateralmente el brazo). Afebril, eupneico, normotenso.
- Cuadro sugiere patología del manguito rotador (tendinopatía/impingement o posible rotura del supraespinoso). Sin datos sistémicos.

2) Banderas rojas:
- Presentes: déficit motor focal (debilidad en abducción).
- Ausentes según lo aportado: dolor torácico/dispnea, fiebre, traumatismo agudo, alteración del nivel de consciencia, dolor en reposo con aspecto séptico, déficit neurológico distal o signos de isquemia de miembro.

3) Nivel de urgencia:
- Urgente (valoración en <60–120 min). Justificación: debilidad objetiva sugiere lesión relevante del manguito rotador (posible rotura) que requiere exploración dirigida e imagen para plan terapéutico y evitar empeoramiento funcional. No hay compromiso vital.

4) Derivación / especialidad recomendada:
- Urgenc

In [20]:
prompt = "Una paciente viene a urgencias, en la revisión se le detecta que la tensión arterial está en 150/95 mmHg, alega trastorno de ansiedad y antecedentes de depresión. El paciente se encuentra muy alterado y no mejora des de hace 24h. Afebril y eupneico."
message = ask_message5(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico:
- Paciente adulta con antecedentes de trastorno de ansiedad y depresión, muy agitada desde hace ~24 h. Afebril y eupneica.
- TA en triage 150/95 mmHg (elevación leve-moderada). Sin otros síntomas referidos.
- Riesgo de causa orgánica subyacente (intoxicación/abstinencia, hipoglucemia, alteración tiroidea) no descartado.

2) Banderas rojas:
- No reportadas por ahora: sin dolor torácico, disnea, fiebre, focalidad neurológica ni alteración del nivel de conciencia descritas.
- A vigilar activamente: agitación psicomotriz persistente (riesgo de auto/heteroagresión), posible ideación suicida/psicosis no explorada, consumo de sustancias.

3) Nivel de urgencia:
- Muy urgente. Justificación: agitación importante mantenida 24 h con necesidad de descartar causas médicas agudas y de valorar riesgo para sí misma o terceros. La TA 150/95 no sugiere emergencia hipertensiva por sí sola.

4) Derivación / especialidad recomendada:
- Urgencias generales para evaluación médica inicial 

In [21]:
prompt = "Hombre de 55 años, con dolor opresivo en el pecho, que se irradia hacia el brazo izquierdo y la mandíbula. Presenta sudoración profusa y sensación de falta de aire. Afebril, tensión elevada."
message = ask_message5(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico:
- Varón de 55 años con dolor torácico opresivo irradiado a brazo izquierdo y mandíbula, sudoración profusa y disnea. Afebril, con tensión elevada.
- Cuadro altamente sugestivo de síndrome coronario agudo (isquemia miocárdica).

2) Banderas rojas:
- Presentes: dolor torácico típico (opresivo con irradiación), disnea, diaforesis. Riesgo cardiovascular (varón >50 años, HTA).

3) Nivel de urgencia:
- Muy urgente. Justificación: sospecha de SCA con síntomas típicos; no se aportan datos de inestabilidad hemodinámica o alteración del nivel de conciencia (si aparecieran, sería crítico/inmediato).

4) Derivación / especialidad recomendada:
- Urgencias generales – circuito de dolor torácico con valoración inmediata por Cardiología. Posible activación de código IAM según ECG.

5) Pruebas iniciales sugeridas:
- ECG de 12 derivaciones en <10 minutos y monitorización continua; repetir ECG seriados si el primero no es diagnóstico.
- Constantes completas: TA, FC, SatO2, FR, Tª; dos

In [22]:
prompt = "Paciente de 60 años, llega al hospital con tos persistente, fiebre y dificultad para respirar. Fue fumador durante 40 años y ha presentado episodios similares en el pasado. Hypertensión moderada."
message = ask_message5(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico:
- Varón de 60 años con tos persistente, fiebre y dificultad respiratoria. Exfumador de larga data (40 años) con episodios previos similares (sugiere exacerbación EPOC/bronquitis crónica) y HTA.
- Cuadro compatible con infección respiratoria baja (neumonía) y/o exacerbación de EPOC; diferenciar también insuficiencia cardiaca o TEP según evolución.

2) Banderas rojas:
- Presentes: disnea, fiebre en >60 años (riesgo de neumonía grave/sepsis, insuficiencia respiratoria).
- Desconocidas por falta de datos: saturación O2, alteración del estado mental, tensión arterial, dolor torácico, cianosis, hemoptisis.

3) Nivel de urgencia:
- Muy urgente. Justificación: disnea + fiebre en adulto mayor con antecedente tabáquico prolongado y episodios previos similares (riesgo de hipoxemia, neumonía, exacerbación EPOC, descompensación cardiaca). Requiere valoración inmediata y monitorización.

4) Derivación / especialidad recomendada:
- Urgencias generales (circuito respiratorio), con 

In [23]:
prompt = "Paciente 45 años, mujer, acude al consultorio con tos persistente y dificultad para respirar. Durante las últimas semanas ha experimentado fatiga extrema y fiebre intermitente. Fumados desde hace 20 años y si tos ha empeorado últimamente. Su historial revela episodios recurrentes de bronquitis. Normotension."
message = ask_message5(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico:
- Mujer 45 años, fumadora desde hace ~20 años, con tos persistente que ha empeorado, disnea, fatiga intensa y fiebre intermitente desde hace semanas. Antecedentes de bronquitis recurrente. Normotensa.
- Cuadro compatible con infección respiratoria (neumonía/bronquitis aguda o reagudización bronquitis crónica/EPOC); por tabaquismo y duración también valorar TB u otra patología pulmonar.

2) Banderas rojas:
- Presentes: disnea, fiebre prolongada (semanas).
- No consta/por confirmar: saturación de O2, taquipnea/taquicardia, dolor torácico pleurítico, hemoptisis, confusión, cianosis.

3) Nivel de urgencia:
- Urgente. Justificación: disnea + fiebre intermitente de semanas en fumadora con empeoramiento de la tos → precisa valoración hoy para descartar neumonía o reagudización infecciosa. No hay datos de inestabilidad hemodinámica, pero faltan constantes clave.

4) Derivación / especialidad recomendada:
- Urgencias generales (Medicina de Urgencias), con posible interconsul

In [24]:
prompt = "Paciente de 25 años, hombre, se presenta en el hospital con síntomas de dolor abdominal intenso, especialmente después de comer, hinchazón y dificultad para defecar. Afebril, eupneico y normotensión."
message = ask_message5(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico: Varón de 25 años con dolor abdominal intenso que empeora tras comer, distensión e dificultad para defecar. Afebril, eupneico y normotenso. El cuadro sugiere posible obstrucción intestinal/íleo o abdomen agudo quirúrgico; diferenciales incluyen cólico biliar, úlcera/pancreatitis, estreñimiento severo.

2) Banderas rojas: Presentes: dolor abdominal intenso + distensión + síntomas obstructivos (dificultad para defecar) → posible obstrucción intestinal/complicación quirúrgica. Ausentes según datos: fiebre, inestabilidad hemodinámica, disnea.

3) Nivel de urgencia: Muy urgente. Justificación: dolor intenso con distensión y síntomas de tránsito sugieren obstrucción/abdomen agudo; requiere evaluación e imagen precoz para descartar isquemia/complicación, pese a constantes estables.

4) Derivación / especialidad recomendada: Urgencias generales con valoración temprana por Cirugía General. Digestivo según hallazgos si se descarta causa quirúrgica (p. ej., biliar/ulcerosa).

5

In [25]:
prompt = "Paciente de 35 años acude a la consulta por alteraciones en su patrón intestinal. Describe cambios en la frecuencia de y la consistencia de las evacuaciones desde hace dos meses. Refiere distensión abdominal persistente y sensación de aumento de presión abdominal. Además reporta dolor durante la defecación y sensación de evacuación incompleta. Manifiesta episodios de náuseas sin vómito y una reciente disminución del apetito. Afebril."
message = ask_message5(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico:
- Adulto de 35 años con 2 meses de cambios en frecuencia y consistencia de las heces, distensión y presión abdominal persistentes, dolor con la defecación y sensación de evacuación incompleta. Náuseas sin vómito, disminución del apetito. Afebril.
- Sin datos aportados de sangrado, pérdida de peso, vómitos, fiebre, ni antecedentes relevantes.

2) Banderas rojas:
- No confirmadas con los datos actuales.
- Potenciales a descartar: sangrado rectal/heces negras, pérdida de peso involuntaria, diarrea nocturna, anemia, historia familiar de cáncer colorrectal/EII, obstrucción (dolor intenso + imposibilidad de expulsar gases/heces).

3) Nivel de urgencia:
- Menos urgente. Justificación: cuadro subagudo (2 meses), afebril, sin vómitos, sin dolor abdominal intenso ni datos de obstrucción o sangrado reportados. Requiere valoración diagnóstica priorizable en consulta en días-semanas, con urgencia solo si aparecen signos de alarma.

4) Derivación / especialidad recomendada:
- Ate

In [26]:
prompt = "Paciente de 60 años, acude al hospital con síntomas de fatiga extrema, presión arterial alta y una hinchazón evidente en las piernas. Afebril, eupneico."
message = ask_message5(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico:
- Varón de 60 años con fatiga extrema, hipertensión y edema evidente en piernas. Afebril y eupneico.
- Sospecha principal: sobrecarga de volumen (insuficiencia cardiaca/renal), menos probable causas hepáticas/endocrinas; a descartar TVP si edema asimétrico.

2) Banderas rojas:
- No reportadas: dolor torácico, disnea en reposo, déficit neurológico, confusión, oliguria/anuria, cefalea intensa/alteración visual, fiebre, dolor/eritema unilateral en pierna.
- A vigilar: TA ≥180/120 mmHg o signos de daño de órgano (torácico, neurológico, renal).

3) Nivel de urgencia:
- Urgente: requiere valoración en horas hoy en Urgencias. Justificación: edema de MMII y fatiga marcada en mayor de 60 años con hipertensión; posible insuficiencia cardiaca o renal que precisa estudio y descartar emergencias hipertensivas.
- Si TA ≥180/120 mmHg o aparece dolor torácico, disnea, déficit neurológico, confusión u oliguria: muy urgente/crítico.

4) Derivación / especialidad recomendada:
- Urgenc

In [27]:
prompt = "Paciente, incapacidad para orinar desde hace 24 horas. Refiere dolor y sensación de plenitud en el abdomen inferior justo con fiebre de 38,5 grados celsius. Sus antecedentes médicos son: Hipertensión, tratada con diuréticos sin infecciones urinarias o problemas renales conocidos. Presenta el abdomen distendido y doloroso a la palpación en la región suprapúbica. Eupneico."
message = ask_message5(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico:
- Anuria/retención urinaria aguda desde hace 24 h, dolor y plenitud suprapúbica, fiebre 38,5 °C. Abdomen distendido y doloroso a la palpación suprapúbica. Eupneico.
- Antecedentes: HTA en tratamiento con diuréticos. Sin IU previas ni enfermedad renal conocida.
- Cuadro compatible con globo vesical con sospecha de infección urinaria asociada (p. ej., prostatitis/ITU complicada) y riesgo de insuficiencia renal/urosepsis.

2) Banderas rojas:
- Presentes: anuria 24 h (obstrucción aguda), fiebre (riesgo de infección/urosepsis), dolor suprapúbico con globo vesical.
- No referidas: hipotensión, confusión, dolor lumbar con déficit neurológico, vómitos incoercibles.

3) Nivel de urgencia:
- Muy urgente. Justificación: requiere descompresión vesical inmediata (sondaje) y descartar/abordar infección potencialmente grave y daño renal agudo. Si aparecieran signos de inestabilidad (hipotensión, confusión, escalofríos intensos), sería crítico.

4) Derivación / especialidad recomen

In [28]:
prompt = "Joven de 17 años, accidente de moto. Ha volado por encima de un coche haciendo una mortal descontrolada. El joven se ha podido levantar y se ha quitado el casco en el momento del accidente. Alega mucho dolor en las piernas, pulsaciones altas que no bajan, dolor en el hombro y en la muñeca. Eupneico, normotensión, afebril"
message = ask_message5(prompt, context, OPENAI_ORG_ID, OPENAI_API_KEY)
print(message)

1) Resumen clínico:
- Varón de 17 años, accidente de moto de alta energía (saltó por encima de un coche). Se quitó el casco tras el impacto.
- Refiere dolor intenso en ambas piernas, hombro y muñeca; taquicardia persistente. Eupneico, normotenso, afebril.
- Politraumatismo potencial con riesgo de lesiones ocultas (cabeza/columna cervical, tórax/abdomen, extremidades).

2) Banderas rojas:
- Presentes: mecanismo de alto impacto; retirada del casco (posible TCE/lesión cervical); taquicardia sostenida; dolor intenso en extremidades (posibles fracturas/compartimental); politrauma.
- No referidas: pérdida de consciencia, vómitos/cefalea intensa, disnea, dolor torácico/abdominal, sangrado masivo.

3) Nivel de urgencia:
- Muy urgente. Justificación: trauma de alta energía con probable lesiones múltiples y taquicardia persistente pese a normotensión; precisa valoración inmediata en box de trauma. Riesgo de deterioro hemodinámico/neurológico.

4) Derivación / especialidad recomendada:
- Urgencia